# Repaired biomedical evidence pipeline
Run this notebook from the project folder with the requirements installed.
The Docker commands in README.md are the primary supported rerun workflow.
This notebook calls the same tested implementation and contains no credentials.
Passage similarity is not a medical confidence score. No training labels are invented.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime, timezone

assert Path("pipeline.py").exists(), "Open the notebook from the biomedical-qa folder"


## Optional: load local configuration without displaying secrets

In [ ]:
config = Path(".env")
if config.exists():
    for line in config.read_text(encoding="utf-8").splitlines():
        if line.strip() and not line.lstrip().startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            if key.strip() in {"NCBI_EMAIL", "NCBI_API_KEY"}:
                os.environ[key.strip()] = value.strip()


## Smoke test, then switch to live data
Set DEMO=False after configuring NCBI_EMAIL in .env. Each execution uses a new output folder.

In [ ]:
DEMO = True
QUERY = "breast cancer"
LIMIT = 10
run_dir = Path("data") / datetime.now(timezone.utc).strftime("notebook-%Y%m%dT%H%M%S%fZ")
command = [sys.executable, "pipeline.py", "run", "--query", QUERY,
           "--limit", str(LIMIT), "--out", str(run_dir)]
if DEMO:
    command.append("--demo")
subprocess.run(command, check=True)


In [ ]:
quality = json.loads((run_dir / "quality.json").read_text(encoding="utf-8"))
quality


## Retrieve cited evidence passages
For a live run, replace the question with your biomedical research question.

In [ ]:
question = "Drug Alpha Condition Beta" if DEMO else "trastuzumab HER2 breast cancer"
subprocess.run([sys.executable, "pipeline.py", "ask", "--data", str(run_dir / "documents.json"),
                "--question", question, "--save", str(run_dir / "answer.json")], check=True)


In [ ]:
answer = json.loads((run_dir / "answer.json").read_text(encoding="utf-8"))
answer
